<a href="https://colab.research.google.com/github/sabihadudhia/Thesis-Hallucination-Benchmarks/blob/main/gpt_oss_20b_HaluEval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import subprocess, os, re, json, time, random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import notebook_login
import warnings, logging

warnings.filterwarnings("ignore")
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
torch.manual_seed(42)

notebook_login()

In [2]:
subprocess.run(["git", "clone", "https://github.com/RUCAIBox/HaluEval.git", "HaluEval"])

CompletedProcess(args=['git', 'clone', 'https://github.com/RUCAIBox/HaluEval.git', 'HaluEval'], returncode=0)

In [3]:
with open('HaluEval/data/qa_data.json') as f:
    all_samples = [json.loads(line) for line in f]

random.seed(42)
sampled = random.sample(all_samples, 300)

instances = []
for s in sampled:
    instances.append({"knowledge": s["knowledge"], "question": s["question"],
                       "answer": s["right_answer"], "is_hallucinated_gt": False})
    instances.append({"knowledge": s["knowledge"], "question": s["question"],
                       "answer": s["hallucinated_answer"], "is_hallucinated_gt": True})

print(f"Total instances: {len(instances)}")

Total instances: 600


In [4]:
model_name = "openai/gpt-oss-20b"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/1.81k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.20k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 27.9MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/36.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

In [5]:
def normalize_response(text):
    return text.strip()

def score_halueval_instance(instance, model, tokenizer):
    prompt = (
        f"Knowledge: {instance['knowledge']}\n"
        f"Question: {instance['question']}\n"
        f"Answer: {instance['answer']}\n"
        f"Is the answer hallucinated? Answer Yes or No only."
    )

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)

    output = model.generate(**inputs, max_new_tokens=300, do_sample=False)
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    response = normalize_response(response).lower()

    yes_positions = [m.start() for m in re.finditer(r'\byes\b', response)]
    no_positions = [m.start() for m in re.finditer(r'\bno\b', response)]

    if not yes_positions and not no_positions:
        return {"correct": None, "raw_response": response, "reason": "unparseable"}

    last_yes = yes_positions[-1] if yes_positions else -1
    last_no = no_positions[-1] if no_positions else -1
    predicted = last_yes > last_no

    return {"correct": predicted == instance["is_hallucinated_gt"], "raw_response": response, "reason": None}

In [6]:
for i in range(10):
    result = score_halueval_instance(instances[i], model, tokenizer)
    print(f"Instance {i}: {result}")

Instance 0: {'correct': True, 'raw_response': 'analysiswe need to determine if the answer "afrojack" is hallucinated. the question: "bebe rexha was a singer who guested on the david guetta song that was produced by which dutch dj?" the song is "hey mama" by david guetta featuring nicki minaj and bebe rexha. the production credits: david guetta, afrojack, and maybe others. the dutch dj is afrojack. so the answer is correct. the answer is not hallucinated. so answer: no.assistantfinalno', 'reason': None}
Instance 1: {'correct': True, 'raw_response': 'analysiswe need to determine if the answer is hallucinated. the question: "bebe rexha was a singer who guested on the david guetta song that was produced by which dutch dj?" the question refers to the david guetta song that bebe rexha guested on. that song is "hey mama" featuring nicki minaj and bebe rexha. the production includes david guetta, nicki minaj, bebe rexha, and afrojack. afrojack is a dutch dj. so the answer should be "afrojack".

In [7]:
from google.colab import drive
from datetime import datetime

drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/drive/MyDrive/thesis_results', exist_ok=True)

save_path = '/content/drive/MyDrive/thesis_results/gptoss20b_halueval_qa_results.jsonl'
print(f"Run started: {datetime.now().isoformat()}")

results = []
start_time = time.time()

with open(save_path, 'w') as f:
    for i, instance in enumerate(instances):
        result = score_halueval_instance(instance, model, tokenizer)
        result["instance_id"] = i
        results.append(result)
        f.write(json.dumps(result) + "\n")
        f.flush()
        os.fsync(f.fileno())
        if i % 50 == 0:
            elapsed = time.time() - start_time
            print(f"Progress: {i}/{len(instances)} | Elapsed: {elapsed:.1f}s")

total_time = time.time() - start_time
print(f"\nDone. Total time: {total_time:.1f}s ({total_time/60:.1f} min)")

valid_results = [r for r in results if r["correct"] is not None]
accuracy = sum(r["correct"] for r in valid_results) / len(valid_results)
invalid_count = len(results) - len(valid_results)

print(f"Accuracy: {accuracy:.3f}")
print(f"Invalid/unparseable responses: {invalid_count} / {len(results)}")

Mounted at /content/drive
Run started: 2026-09-17T13:30:29.436097
Progress: 0/600 | Elapsed: 6.1s
Progress: 50/600 | Elapsed: 442.1s
Progress: 100/600 | Elapsed: 842.8s
Progress: 150/600 | Elapsed: 1233.4s
Progress: 200/600 | Elapsed: 1572.1s
Progress: 250/600 | Elapsed: 1964.0s
Progress: 300/600 | Elapsed: 2356.3s
Progress: 350/600 | Elapsed: 2741.5s
Progress: 400/600 | Elapsed: 3144.6s
Progress: 450/600 | Elapsed: 3503.7s
Progress: 500/600 | Elapsed: 3921.6s
Progress: 550/600 | Elapsed: 4315.5s

Done. Total time: 4716.4s (78.6 min)
Accuracy: 0.864
Invalid/unparseable responses: 35 / 600
